# Detecting mechanical registration errors in OMR answer sheets

A candidate skips a bubble row. Every answer after that sits one row too low. All
of them are correct. All of them are marked wrong.

This notebook runs three things in under a minute: the case that prompted the
work, why the obvious generous approach fails, and the detector recovering a
planted error. The full report, benchmark and results are in the repository.

Run the cells in order.


## Setup

Runs in Colab or locally. In Colab it clones the repository; locally it uses the files already present.


In [ ]:
# Opening this notebook from GitHub in Colab does not bring the code with it,
# so fetch the repository first.
import os, sys, time

REPO = "https://github.com/Enayat-Hassani/omr-shift-detection"

if not os.path.exists("omr_shift.py"):
    name = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(name):
        os.system(f"git clone --depth 1 {REPO} 2>/dev/null")
    if os.path.isdir(name):
        os.chdir(name)

if not os.path.exists("omr_shift.py"):
    raise SystemExit(
        "omr_shift.py not found. Either set REPO above to the repository URL, "
        "or upload the .py files with:  from google.colab import files; files.upload()")

sys.path.insert(0, ".")
from omr_shift import *
print("ready:", os.getcwd())

## 1. The sheet in question

46 maths questions. The student scored 7. Their other subjects were reportedly
fine, which is why somebody suspected a shift.

In [ ]:
sheet = ResponseSheet.from_records(
    DATA["evaluation_data"], candidate_id="CANDIDATE-001", subject="Mathematics")
print("key    :", "".join(sheet.key))
print("student:", "".join(m or '.' for m in sheet.marks))
print(f"\nscore as marked: {sheet.raw_score()} out of {sheet.n_questions}")
print(f"random guessing would give about {sheet.n_questions/4:.0f}")

## 2. Why the obvious approach does not work

The generous way to handle shifts is to find the longest run of the student's
answers that appears in the key in the same order, and award that. It needs no
settings and handles shifts automatically.

Watch what it does to a sheet filled in at random.

In [ ]:
import random
def lcs(a, b):
    prev = [0]*(len(b)+1)
    for x in a:
        cur = [0]*(len(b)+1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j-1]+1 if x == y else max(prev[j], cur[j-1])
        prev = cur
    return prev[-1]

rng = random.Random(1)
strict   = sheet.raw_score()
generous = lcs(sheet.marks, sheet.key)
rand = [lcs([rng.choice("ABCD") for _ in range(46)], sheet.key) for _ in range(400)]

print(f"this student, marked strictly      : {strict:5} / 46")
print(f"this student, generous method      : {generous:5} / 46")
print(f"a RANDOM sheet, generous method    : {sum(rand)/len(rand):7.1f} / 46")
print()
print("The generous method hands out 23 marks for answers that score")
print("worse than random guessing under its own rule.")

## 3. What our method says about this sheet

Every check has to pass before anything is corrected.

In [ ]:
t = time.time()
adj = Adjudicator(sheet, AdjudicationConfig()).run(n_permutations=4000, verbose=False)
print(f"({time.time()-t:.1f} seconds)\n")
print(f"VERDICT: {adj.verdict}\n")
print(f"score as marked : {adj.raw_score} / 46")
print(f"score after     : {adj.adjudicated_score} / 46\n")
for name, g in adj.gates.items():
    print(f"  [{'pass' if g['passed'] else 'FAIL'}] {name}")
print()
w = adj.calibration["scan_window"]
if w:
    print(f"Best run of correct answers at any shifted position:")
    print(f"  questions {w['q_start']} to {w['q_end']}, shifted by {w['offset']:+d}, "
          f"{w['n_correct']} right out of {w['n_items']}")
print(f"Sheets known to contain no shift do that well or better "
      f"{adj.calibration['p_value']:.0%} of the time.")

## 4. Does the detector actually catch real shifts

A null result means nothing unless the detector works. Same answer key, a student
who knows 85% of the material, one bubble row skipped at question 16.

In [ ]:
ctrl = demo_planted_shift(AdjudicationConfig(), '.')
print(f"VERDICT: {ctrl.verdict}\n")
print(f"score as marked : {ctrl.raw_score} / 46")
print(f"score after     : {ctrl.adjudicated_score} / 46")
gained = sum(1 for r in ctrl.item_ledger if r['change'] == 'GAIN')
lost   = sum(1 for r in ctrl.item_ledger if r['change'] == 'LOSS')
print(f"marks gained {gained}, marks lost {lost}   (losses count too)\n")
for c in ctrl.change_points:
    print(f"  found at question {c['at_question']}: "
          f"shift {c['offset_before']:+d} -> {c['offset_after']:+d}")

## 5. One question at a time

Even after all checks pass, a question is only re-read if we are at least 99%
sure which row it belongs to. Questions near the edge of the shift are left
alone.

In [ ]:
for r in ctrl.item_ledger[13:22]:
    mark = "yes" if r['final_correct'] else "no "
    print(f"  Q{r['question']:>2}  key {r['key']}  confidence {r['map_posterior']:.3f}  "
          f"correct now: {mark}  {r['reason'][:52]}")

## 6. Benchmark results

Ten candidate models, each built to violate a different assumption, and five
detectors evaluated on identical sheets. These figures are read from the
committed results. The full run takes several minutes.

In [ ]:
import json

R = json.load(open("results/benchmark.json"))
order = ["no-op (never correct)", "brute-force shift", "LCS (maximally generous)",
         "fixed-cost DP alignment", "gated pair-HMM (reference)"]
noop = sum(r["marks_wrongly_withheld"] for r in R
           if r["detector"] == order[0]) / len([r for r in R if r["detector"] == order[0]])

print(f"{'detector':<28}{'worst FPR':>11}{'awarded':>10}{'recovery':>10}{'Brier':>9}")
print("-" * 68)
for d in order:
    rs = [r for r in R if r["detector"] == d]
    br = [r["brier"] for r in rs if r["brier"] is not None]
    hold = sum(r["marks_wrongly_withheld"] for r in rs) / len(rs)
    print(f"{d:<28}{max(r['fpr'] for r in rs):>11.2f}"
          f"{sum(r['marks_wrongly_awarded'] for r in rs)/len(rs):>10.2f}"
          f"{(noop-hold)/noop:>9.0%}"
          f"{(f'{sum(br)/len(br):.3f}' if br else 'none'):>9}")

print()
print("worst FPR is the highest false positive rate across the ten candidate models.")
print("awarded is marks given that were not earned. recovery is the share of marks")
print("lost to an error that the detector returns. Brier measures whether the")
print("reported confidence is trustworthy; lower is better, and it is undefined")
print("for detectors that report no confidence at all.")

## 7. Figures

Two of the seven committed figures. The first shows the null calibration for the
case sheet: the observed evidence sits inside the bulk of what sheets containing
no error produce. The second shows the positive control, where a planted error
produces a clean change in registration with genuine uncertainty at its edges.

In [ ]:
from IPython.display import Image, display

for path, caption in [
    ("results/figures/4_null_calibration.png",
     "Case sheet: observed evidence against three null models"),
    ("results/figures/positive_control/1_displacement_posterior.png",
     "Positive control: per-question posterior over displacement"),
]:
    print(caption)
    display(Image(path))

## Reading further

All of these are in the repository and need no execution.

| | |
|---|---|
| `REPORT.md` | Problem, models evaluated, comparison, recommended configuration, safeguards |
| `ASSUMPTIONS.md` | Eleven claims made during development that measurement contradicted |
| `CASE_REPORT.md` | Full analysis of the sheet in section 1 |
| `results/case_default_prior.txt` | Every question of that sheet, with the reasoning |
| `results/benchmark.txt` | Per-generator benchmark tables |

The benchmark itself is `benchmark/omrbench.py` and takes several minutes, which
is why its output is committed here.